# ACTIVE-TB-S: Post-Freeze Official Evaluation Pilot

This phase receives hash-frozen testbenches plus the VerilogEval golden RTL, official testbenches, and released CorrectBench mutant bank. It cannot modify the generated suites.


In [ ]:
from pathlib import Path
import json, shutil, subprocess, sys, zipfile
INPUT=Path('/kaggle/input'); WORK=Path('/kaggle/working')
markers=list(INPUT.rglob('active_tb_s_evaluation_pilot_v1/EVALUATION_BOUNDARY.json'))
if len(markers)==1: SOURCE=markers[0].parent
else:
    archives=list(INPUT.rglob('active_tb_s_evaluation_pilot_v1.zip')); assert len(archives)==1, archives
    dst=WORK/'attached_evaluation_source'
    with zipfile.ZipFile(archives[0]) as zf: zf.extractall(dst)
    SOURCE=dst/'active_tb_s_evaluation_pilot_v1'
PROJECT=WORK/'active_tb_s_evaluation_project'
if PROJECT.exists(): shutil.rmtree(PROJECT)
shutil.copytree(SOURCE/'project',PROJECT)
FROZEN=WORK/'active_tb_s_frozen_pilot_v1'
frozen_archive=SOURCE/'active_tb_s_frozen_pilot_v1.zip'
frozen_directory=SOURCE/'active_tb_s_frozen_pilot_v1'
if frozen_archive.exists():
    with zipfile.ZipFile(frozen_archive) as zf: zf.extractall(FROZEN)
elif frozen_directory.is_dir():
    shutil.copytree(frozen_directory,FROZEN)
else:
    raise FileNotFoundError({'archive':str(frozen_archive),'directory':str(frozen_directory),'source_files':[p.name for p in SOURCE.iterdir()]})
OUT=WORK/'active_tb_s_evaluation_pilot_v1'; OUT.mkdir(exist_ok=True)
print('Evaluation source:',SOURCE)


In [ ]:
boundary=json.loads((SOURCE/'EVALUATION_BOUNDARY.json').read_text())
assert boundary['frozen_before_golden_access'] is True
manifest=json.loads((FROZEN/'FROZEN_BEFORE_EVALUATION.json').read_text())
assert manifest['generation_input_fields']==['task_id','description','header']
print('POST_FREEZE_PROVENANCE_AUDIT_PASS:',boundary['task_count'],'tasks')


In [ ]:
subprocess.run(['apt-get','update','-qq'],check=True)
subprocess.run(['apt-get','install','-y','-qq','iverilog'],check=True)
cmd=[sys.executable,str(PROJECT/'run_spec_only_evaluation.py'),
 '--frozen-root',str(FROZEN/'frozen'),
 '--tasks',str(SOURCE/'HDLBits_pilot.jsonl'),
 '--mutants',str(SOURCE/'HDLBits_mutants_pilot.jsonl'),
 '--output',str(OUT),'--timeout','30','--resume']
subprocess.run(cmd,check=True,cwd=PROJECT)


In [ ]:
import pandas as pd
aggregate=pd.read_csv(OUT/'spec_only_eval2_aggregate.csv')
display(aggregate)
per_task=pd.read_csv(OUT/'spec_only_eval2_per_task.csv')
pivot=per_task.pivot(index='task_id',columns='method',values='eval2_ratio')
display(pivot.describe().T)
archive=WORK/'active_tb_s_evaluation_pilot_v1.zip'
shutil.make_archive(str(archive.with_suffix('')),'zip',OUT)
print('Evaluation checkpoint:',archive,archive.stat().st_size,'bytes')
